# AI CCTV Surveillance — Phase 1 POC (GPU)
## Kandhamal District Police | Starlight Data Solutions

**Run this notebook on Google Colab with GPU runtime.**

Runtime → Change runtime type → GPU (T4)

This notebook runs all 3 Phase 1 use cases:
1. **UC-1:** Stolen Vehicle Detection (ANPR)
2. **UC-2:** Case-Linked Vehicle Alert
3. **UC-3:** Face Recognition — Wanted Persons

---
## Step 1: Check GPU & Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch GPU available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Install all dependencies
!pip install -q ultralytics easyocr face_recognition deepface fastapi uvicorn python-multipart pyngrok nest-asyncio pillow opencv-python-headless
print("All dependencies installed!")

---
## Step 2: Clone Repo & Download Models

In [ ]:
# Clone the repo
!git clone https://github.com/saiesh2401/odisha-dist-cctv.git
%cd odisha-dist-cctv/poc

# Download plate detector model
!git clone --depth 1 https://github.com/sveyek/Video-ANPR.git /tmp/video-anpr
!cp /tmp/video-anpr/models/license_plate_detector.pt models/
!rm -rf /tmp/video-anpr

# Create required directories
!mkdir -p results uploads wanted_persons

print("\nRepo cloned, models ready!")
!ls -la models/

---
## Step 3: Initialize AI Engine

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

from engine import SurveillanceEngine

engine = SurveillanceEngine()

# Force load all models now (with GPU)
engine.anpr._ensure_models()
print("\nAll AI models loaded and ready!")

---
## Step 4: Test UC-1 & UC-2 — ANPR (Stolen Vehicle + Case-Linked)

### 4a: Test Database Matching (instant)

In [ ]:
# Test plate matching against all databases
test_plates = [
    "OD02AK7834",  # Stolen vehicle
    "OD21F9012",   # Stolen vehicle
    "MH12AB1234",  # Stolen vehicle
    "OD21C7890",   # Case-linked (Robbery)
    "WB02X5678",   # Case-linked + Watchlist (out-of-state)
    "OD21A1111",   # Watchlist only
    "OD05M8901",   # Stolen vehicle
    "OD21D9999",   # Clean — no match
]

print("Plate Number      | Result")
print("-" * 60)
for plate in test_plates:
    result = engine.anpr.match_plate(plate)
    if result['matches']:
        for m in result['matches']:
            print(f"{plate:18s}| {m['severity']:8s} → {m['type']}")
    else:
        print(f"{plate:18s}| CLEAR — no match")

### 4b: Test ANPR on an Image

Upload any image with a vehicle, or use the included test images.

In [ ]:
import cv2
import json
from IPython.display import Image, display

# Test with included demo image
test_image = "data/demo_OD02AK7834.jpg"

if os.path.exists(test_image):
    result = engine.process_anpr(test_image)
    print(f"Vehicles detected: {result.get('vehicles_detected', 0)}")
    print(f"Plates detected: {result.get('plates_detected', 0)}")
    print(f"Plates read: {result.get('plates_read', 0)}")
    print(f"Alerts: {len(result.get('alerts', []))}")
    print(f"Processing time: {result.get('processing_time_sec', 0)}s")
    
    if result.get('annotated_image'):
        display(Image(filename=result['annotated_image']))
    
    for a in result.get('alerts', []):
        for m in a.get('matches', []):
            print(f"\nALERT [{m['severity']}]: {m['type']}")
            print(f"  Plate: {a['plate']}")
            print(f"  Details: {json.dumps(m['details'], indent=2)}")
else:
    print("Upload an image with a vehicle to test.")
    print("Use the file upload button in the sidebar, or:")
    print('from google.colab import files; uploaded = files.upload()')

### 4c: Upload Your Own Image

In [ ]:
from google.colab import files
from IPython.display import Image, display

print("Upload an image with a vehicle (any traffic photo / CCTV screenshot):")
uploaded = files.upload()

for filename in uploaded:
    with open(f"uploads/{filename}", 'wb') as f:
        f.write(uploaded[filename])
    
    print(f"\nProcessing: {filename}")
    result = engine.process_anpr(f"uploads/{filename}")
    
    print(f"Vehicles: {result.get('vehicles_detected', 0)}")
    print(f"Plates: {result.get('plates_read', 0)}")
    print(f"Alerts: {len(result.get('alerts', []))}")
    print(f"Time: {result.get('processing_time_sec', 0)}s")
    
    if result.get('annotated_image'):
        display(Image(filename=result['annotated_image']))
    
    for p in result.get('plate_details', []):
        print(f"  Plate: {p['plate_text']}")
    
    for a in result.get('alerts', []):
        for m in a.get('matches', []):
            print(f"  ALERT [{m['severity']}]: {m['type']} — {a['plate']}")

### 4d: Test ANPR on Video (Upload CCTV footage)

In [ ]:
from google.colab import files
import time

print("Upload a traffic video (.mp4):")
uploaded = files.upload()

for filename in uploaded:
    video_path = f"uploads/{filename}"
    with open(video_path, 'wb') as f:
        f.write(uploaded[filename])
    
    print(f"\nProcessing video: {filename}")
    t0 = time.time()
    result = engine.anpr.process_video(video_path, sample_every_n_frames=15)
    elapsed = time.time() - t0
    
    print(f"\n{'='*50}")
    print(f"RESULTS")
    print(f"{'='*50}")
    print(f"Frames processed: {result['frames_processed']} / {result['total_frames']}")
    print(f"Processing time: {elapsed:.1f}s")
    print(f"Processing FPS: {result['fps_processed']}")
    print(f"Unique plates: {len(result['unique_plates_detected'])}")
    
    print(f"\nPlates detected:")
    for p in result['unique_plates_detected']:
        print(f"  {p}")
    
    print(f"\nAlerts: {len(result['alerts'])}")
    for a in result['alerts']:
        for m in a['matches']:
            print(f"  [{m['severity']}] {m['type']}: {a['plate']} @ {a['timestamp_sec']}s")
    
    # Show alert evidence images
    for a in result['alerts']:
        if 'evidence_image' in a:
            print(f"\nEvidence for {a['plate']}:")
            display(Image(filename=a['evidence_image'], width=600))

---
## Step 5: Test UC-3 — Face Recognition

### 5a: Enroll a Wanted Person

In [ ]:
from google.colab import files

print("Upload a face photo to ENROLL as a wanted person:")
uploaded = files.upload()

for filename in uploaded:
    name = input("Enter name for this person: ")
    safe_name = name.replace(" ", "_")
    dest = f"wanted_persons/{safe_name}.jpg"
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f"Saved: {dest}")

# Reload database
count = engine.face.load_wanted_persons()
print(f"\nTotal wanted persons enrolled: {count}")

### 5b: Search for a Face Match

In [ ]:
from google.colab import files
from IPython.display import Image, display

print(f"Wanted persons in database: {len(engine.face.wanted_persons)}")
for name in engine.face.wanted_persons:
    print(f"  - {name}")

print("\nUpload a face photo to SEARCH against the database:")
uploaded = files.upload()

for filename in uploaded:
    save_path = f"uploads/{filename}"
    with open(save_path, 'wb') as f:
        f.write(uploaded[filename])
    
    print(f"\nSearching: {filename}")
    result = engine.face.search_face(save_path)
    
    print(f"Faces detected: {result['faces_detected']}")
    print(f"Processing time: {result['processing_time_sec']}s")
    
    if result['matches']:
        for m in result['matches']:
            severity_color = '\033[91m' if m['severity'] == 'CRITICAL' else '\033[93m' if m['severity'] == 'REVIEW' else '\033[0m'
            print(f"\n{severity_color}[{m['severity']}] MATCH FOUND\033[0m")
            print(f"  Person: {m['matched_person']}")
            print(f"  Confidence: {m['confidence']}%")
            print(f"  Distance: {m['distance']}")
            print(f"  DB Photo: {m['db_photo']}")
    else:
        print("\nNo matches found in wanted persons database.")

---
## Step 6: Run Full Acceptance Test (25 criteria)

In [ ]:
!python generate_acceptance_report.py

---
## Step 7: Launch Web Demo Server (with public URL)

This starts the FastAPI server and creates a public ngrok URL you can open on any device.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn

# Set your ngrok auth token (get free at https://dashboard.ngrok.com)
# ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")

# Create tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"  POC Demo Server is LIVE!")
print(f"  Public URL: {public_url}")
print(f"  Open this URL on any device to demo.")
print(f"{'='*60}\n")

# Import and run server
from server import app
uvicorn.run(app, host="0.0.0.0", port=8000)

---
## Notes

- **GPU**: This notebook uses Google Colab's free T4 GPU (16GB VRAM)
- **Models**: YOLOv8n + Plate Detector + EasyOCR + dlib face_recognition
- **All open source**: Zero license cost
- **ngrok**: Creates a public URL so you can demo from your phone
- **Runtime**: Select GPU runtime for best performance (Runtime → Change runtime type → GPU)